In [94]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/heart.csv")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


In [95]:
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [96]:
df.duplicated().sum()

np.int64(0)

In [97]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,918.0,53.510893,9.432617,28.0,47.00,54.0,60.0,77.0
RestingBP,918.0,132.396514,18.514154,0.0,120.00,130.0,140.0,200.0
Cholesterol,918.0,198.799564,109.384145,0.0,173.25,223.0,267.0,603.0
FastingBS,918.0,0.233115,0.423046,0.0,0.00,0.0,0.0,1.0
MaxHR,918.0,136.809368,25.460334,60.0,120.00,138.0,156.0,202.0
Oldpeak,918.0,0.887364,1.066570,-2.6,0.00,0.6,1.5,6.2
HeartDisease,918.0,0.553377,0.497414,0.0,0.00,1.0,1.0,1.0


In [98]:
df.describe(include="object").T

,count,unique,top,freq
Sex,918,2,M,725
ChestPainType,918,4,ASY,496
RestingECG,918,3,Normal,552
ExerciseAngina,918,2,N,547
ST_Slope,918,3,Flat,460


In [99]:
num_cols = df.select_dtypes("number").columns.to_list()
cat_cols = df.select_dtypes("object").columns.to_list()
full_cat_cols = cat_cols + ["FastingBS", "HeartDisease"]

print(f"Num cols: {num_cols}")
print(f"Cat cols: {cat_cols}")

Num cols: ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak', 'HeartDisease']
Cat cols: ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']


In [100]:
for col in full_cat_cols:
    print(f"{col}: {df[col].unique()}")

Sex: ['M' 'F']
ChestPainType: ['ATA' 'NAP' 'ASY' 'TA']
RestingECG: ['Normal' 'ST' 'LVH']
ExerciseAngina: ['N' 'Y']
ST_Slope: ['Up' 'Flat' 'Down']
FastingBS: [0 1]
HeartDisease: [0 1]


### Encoding

In [101]:
def label_encode(df: pd.DataFrame):
    struct = dict()

    for col in cat_cols:
        unq = df[col].unique()
        unq_map = {v: k for k, v in enumerate(unq)}
        struct[col] = unq_map
        df[col] = df[col].map(unq_map)
    
    return df, struct

In [102]:
df_enc, struct = label_encode(df)
df_enc.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,0,0,140,289,0,0,172,0,0.0,0,0
1,49,1,1,160,180,0,0,156,0,1.0,1,1
2,37,0,0,130,283,0,1,98,0,0.0,0,0
3,48,1,2,138,214,0,0,108,1,1.5,1,1
4,54,0,1,150,195,0,0,122,0,0.0,0,0


In [103]:
struct

{'Sex': {'M': 0, 'F': 1},
 'ChestPainType': {'ATA': 0, 'NAP': 1, 'ASY': 2, 'TA': 3},
 'RestingECG': {'Normal': 0, 'ST': 1, 'LVH': 2},
 'ExerciseAngina': {'N': 0, 'Y': 1},
 'ST_Slope': {'Up': 0, 'Flat': 1, 'Down': 2}}

### Spliting Data

In [ ]:
def train_test_split(X: pd.DataFrame, y: pd.DataFrame, test_size=0.2, random_state=None):
    if random_state:
        np.random.seed(random_state)
    
    n_samples = len(X)
    shuffle_indices = np.random.permutation(np.arange(n_samples))

    test_size = round(test_size * n_samples)
    train_size = test_size - n_samples

    train_indices = shuffle_indices[:train_size]
    test_indices = shuffle_indices[test_size:]  

    X_train, X_test = X.iloc[train_indices], X.iloc[test_indices]
    y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]

    return [X_train, X_test, y_train, y_test]

In [105]:
X = df.drop(columns=["HeartDisease"])
y = df.HeartDisease

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
668,63,1,0,140,195,0,0,179,0,0.0,0
30,53,0,1,145,518,0,0,130,0,0.0,1
377,65,0,2,160,0,1,1,122,0,1.2,1
535,56,0,2,130,0,0,2,122,1,1.0,1
807,54,0,0,108,309,0,0,156,0,0.0,0


### Scalers

In [106]:
def min_max_scaler(X: pd.DataFrame):
    return ((X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0)))

def std_scaler(X: pd.DataFrame):
    return ((X - X.mean(axis=0)) - X.std(axis=0))

In [107]:
X_train_mm_scaler = min_max_scaler(X_train)
X_test_mm_scaler = min_max_scaler(X_test)

X_train_mm_scaler.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
668,0.729167,1.0,0.000000,0.500000,0.345745,0.0,0.0,0.870968,0.0,0.215686,0.0
30,0.520833,0.0,0.333333,0.541667,0.918440,0.0,0.0,0.475806,0.0,0.215686,0.5
377,0.770833,0.0,0.666667,0.666667,0.000000,1.0,0.5,0.411290,0.0,0.450980,0.5
535,0.583333,0.0,0.666667,0.416667,0.000000,0.0,1.0,0.411290,1.0,0.411765,0.5
807,0.541667,0.0,0.000000,0.233333,0.547872,0.0,0.0,0.685484,0.0,0.215686,0.0


In [108]:
X_train_std_scaler = std_scaler(X_train)
X_test_std_scaler = std_scaler(X_test)

X_train_std_scaler.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
668,0.342693,0.49845,-2.364428,-8.357957,-114.42963,-0.69273,-1.438563,13.780433,-0.853569,-1.815092,-1.239326
30,-9.657307,-0.50155,-1.364428,-3.357957,208.57037,-0.69273,-1.438563,-35.219567,-0.853569,-1.815092,-0.239326
377,2.342693,-0.50155,-0.364428,11.642043,-309.42963,0.30727,-0.438563,-43.219567,-0.853569,-0.615092,-0.239326
535,-6.657307,-0.50155,-0.364428,-18.357957,-309.42963,-0.69273,0.561437,-43.219567,0.146431,-0.815092,-0.239326
807,-8.657307,-0.50155,-2.364428,-40.357957,-0.42963,-0.69273,-1.438563,-9.219567,-0.853569,-1.815092,-1.239326


### Build Model

In [ ]:
class KNN:
    def __init__(self, k: int, p: int, weight, random_state=None):
        self.k = k
        self.p = p
        self.weight = weight
        self.random_state = random_state

    def fit(self, X_train: pd.DataFrame, y_train: pd.DataFrame):
        self.X_train = X_train.to_numpy()
        self.y_train = y_train.to_numpy()
    
    def _minkowski_distance(self, a, b):
        return np.power(np.sum(np.abs(a - b), axis=1), 1/self.p)
    
    def _get_nn(self, X_test: pd.DataFrame):
        distance = self._minkowski_distance(X_test, self.X_train)
        k_idxs = np.argsort(distance)[:self.k]
        k_labels = [self.y_train[i] for i in k_idxs]
        k_distance = [distance[i] for i in k_idxs]


        if self.weight == "uniform":
            values, counts = np.unique(k_labels, return_counts=True)
            max_count = max(counts)
            candidates = values[counts == max_count]
            return np.random.choice(candidates)
        
        elif self.weight == "distance":
            weights = [(1 / (d + 1e-5)) for d in k_distance]
        
        elif callable(self.weight):
            weights = [self.weight(d) for d in k_distance]
        
        label_weight = dict()
        for label, weight in zip(k_labels, weights):
            label_weight[label] = label_weight.get(label, 0) + weight
        
        max_weight = max(label_weight.values())
        candidates = [
            label 
            for label, weight in label_weight.items() 
            if weight == max_weight
        ]

        return np.random.choice(candidates)
    
    def predict(self, X_test: pd.DataFrame):
        return np.array([
            self._get_nn(x)
            for x in X_test.to_numpy()
        ])

### Evaluation Model

In [110]:
class EvaluationModel:
    def __init__(self, y_true: pd.DataFrame, y_pred: np.ndarray):
        self.y_true = y_true
        self.y_pred = y_pred
    
    def accuracy(self):
        return np.sum(self.y_true == self.y_pred) / len(self.y_true)
    
    def precision(self, TP, FP):
        return TP / (TP + FP) if (TP + FP) != 0 else 0
    
    def recall(self, TP, FN):
        return TP / (TP + FN) if (TP + FN) != 0 else 0
    
    def f1_score(self, TP, FP, FN):
        P = self.precision(TP, FP)
        R = self.precision(TP, FN)

        return 2 * ((P * R) / (P + R))
    
    def macro_avg(self, report_df: pd.DataFrame):
        return report_df[["Precision", "Recall", "F1-Score"]].mean()
    
    def weighted_avg(self, report_df: pd.DataFrame):
        weights = np.bincount(self.y_true)
        total = weights.sum()

        return (report_df[["Precision", "Recall", "F1-Score"]].T * weights / total).T.sum()
    
    def make_report(self):
        cats = np.unique(self.y_true)
        reports = {"Category": [], "Precision": [], "Recall": [], "F1-Score": []}

        for cat in cats:
            # Confusion Matrix
            TP = np.sum((self.y_true == cat) & (self.y_pred == cat)) # Asli POSITIF prediksi POSITIF
            TN = np.sum((self.y_true != cat) & (self.y_pred != cat)) # Asli NEGATIF prediksi NEGATIF
            FP = np.sum((self.y_true != cat) & (self.y_pred == cat)) # Asli NEGATIF prediksi POSITIF
            FN = np.sum((self.y_true == cat) & (self.y_pred != cat)) # Asli POSITIF prediksi NEGATIF

            # Calculation Matric
            P = self.precision(TP, FP)
            R = self.recall(TP, FN)
            F1 = self.f1_score(TP, FP, FN)

            # Append Matrixs
            reports["Category"].append(cat)
            reports["Precision"].append(P)
            reports["Recall"].append(R)
            reports["F1-Score"].append(F1)
        
        accuracy = self.accuracy()
        report_df = pd.DataFrame(reports)

        macro_avg = self.macro_avg(report_df)
        weighted_avg = self.weighted_avg(report_df)

        result_df = pd.DataFrame.from_records([
            {"Category": "macro avg", **macro_avg},
            {"Category": "weighted avg", **weighted_avg}
        ])

        final_report_df = pd.concat([result_df, result_df], ignore_index=True)

        return final_report_df, accuracy

### Search Good Model

In [111]:
def log_train(param_result, index, total):
    progress_text = f"[{index}/{total}] [{index/total*100:.2f}%]"
    separator = "="*100

    print(separator)
    print(progress_text)
    print(param_result)
    print(separator)
    

def search_good_model(params: dict, limit_data: int = None):
    total_loop = (
        len(params["K"]) *
        len(params["P"]) *
        len(params["weight"]) *
        len(params["random_state"]) *
        len(params["Scaler"])
    )
    count = 1

    models = list()

    for k in params["K"]:
        for p in params["P"]:
            for weight in params["weight"]:
                for rs in params["random_state"]:
                    for i, (train, test) in enumerate(params["Scaler"]):
                        # Initalization KNN model
                        model = KNN(k=k, p=p, weight=weight, random_state=rs)
                        model.fit(X_train=train[:limit_data], y_train=y_train[:limit_data])

                        preds = model.predict(X_test=test[:limit_data])

                        # Evaluation
                        eval_model = EvaluationModel(y_true=y_test[:limit_data], y_pred=preds)
                        report_df, accuracy = eval_model.make_report()

                        # Append
                        params_result = {
                            "K": k,
                            "P": p,
                            "weight": weight,
                            "random_state": rs,
                            "Scaler": i
                        }

                        log_train(params_result, count, total_loop)

                        models.append({
                            "params": params_result,
                            "Accuracy": accuracy,
                            "report_df": report_df
                        })

                        count += 1
    
    return models

In [112]:
def custom_distance(d):
    return 1 / (d ** 2 + 1e-5)

params = {
    "K": range(3, 9),
    "P": range(1, 5),
    "weight": ["uniform", "distance", custom_distance],
    "random_state": [0, 44, 99],
    "Scaler": [(X_train_mm_scaler, X_test_mm_scaler), (X_train_std_scaler, X_test_std_scaler), (X_train, X_test)]
}

models = search_good_model(params)

[1/648] [0.15%]
{'K': 3, 'P': 1, 'weight': 'uniform', 'random_state': 0, 'Scaler': 0}
[2/648] [0.31%]
{'K': 3, 'P': 1, 'weight': 'uniform', 'random_state': 0, 'Scaler': 1}
[3/648] [0.46%]
{'K': 3, 'P': 1, 'weight': 'uniform', 'random_state': 0, 'Scaler': 2}
[4/648] [0.62%]
{'K': 3, 'P': 1, 'weight': 'uniform', 'random_state': 44, 'Scaler': 0}
[5/648] [0.77%]
{'K': 3, 'P': 1, 'weight': 'uniform', 'random_state': 44, 'Scaler': 1}
[6/648] [0.93%]
{'K': 3, 'P': 1, 'weight': 'uniform', 'random_state': 44, 'Scaler': 2}
[7/648] [1.08%]
{'K': 3, 'P': 1, 'weight': 'uniform', 'random_state': 99, 'Scaler': 0}
[8/648] [1.23%]
{'K': 3, 'P': 1, 'weight': 'uniform', 'random_state': 99, 'Scaler': 1}
[9/648] [1.39%]
{'K': 3, 'P': 1, 'weight': 'uniform', 'random_state': 99, 'Scaler': 2}
[10/648] [1.54%]
{'K': 3, 'P': 1, 'weight': 'distance', 'random_state': 0, 'Scaler': 0}
[11/648] [1.70%]
{'K': 3, 'P': 1, 'weight': 'distance', 'random_state': 0, 'Scaler': 1}
[12/648] [1.85%]
{'K': 3, 'P': 1, 'weight': 

In [113]:
from IPython.display import display

sorted_models = sorted(models, key=lambda x: x["Accuracy"], reverse=True)[:3]

for model in sorted_models:
    print(f"Params: {model["params"]}")
    display(model["report_df"])
    print(f"Accuracy: {model["Accuracy"]}")

Params: {'K': 6, 'P': 4, 'weight': 'uniform', 'random_state': 0, 'Scaler': 0}


,Category,Precision,Recall,F1-Score
0,macro avg,0.866392,0.859016,0.861389
1,weighted avg,0.864949,0.863760,0.863069
2,macro avg,0.866392,0.859016,0.861389
3,weighted avg,0.864949,0.863760,0.863069


Accuracy: 0.8637602179836512
Params: {'K': 6, 'P': 2, 'weight': 'distance', 'random_state': 0, 'Scaler': 0}


,Category,Precision,Recall,F1-Score
0,macro avg,0.864368,0.858024,0.860153
1,weighted avg,0.863233,0.862398,0.861794
2,macro avg,0.864368,0.858024,0.860153
3,weighted avg,0.863233,0.862398,0.861794


Accuracy: 0.8623978201634878
Params: {'K': 6, 'P': 2, 'weight': 'distance', 'random_state': 44, 'Scaler': 0}


,Category,Precision,Recall,F1-Score
0,macro avg,0.864368,0.858024,0.860153
1,weighted avg,0.863233,0.862398,0.861794
2,macro avg,0.864368,0.858024,0.860153
3,weighted avg,0.863233,0.862398,0.861794


Accuracy: 0.8623978201634878


### Use model with good params

In [116]:
final_model = KNN(k=6, p=4, weight="uniform", random_state=0)
final_model.fit(X_train=X_train_mm_scaler, y_train=y_train)

preds = final_model.predict(X_test=X_test_mm_scaler)

# Evaluation
eval_final_model = EvaluationModel(y_true=y_test, y_pred=preds)
report_df, accuracy = eval_final_model.make_report()

display(report_df)
print(f"Accuracy: {accuracy}")

,Category,Precision,Recall,F1-Score
0,macro avg,0.865208,0.857515,0.859952
1,weighted avg,0.863691,0.862398,0.861667
2,macro avg,0.865208,0.857515,0.859952
3,weighted avg,0.863691,0.862398,0.861667


Accuracy: 0.8623978201634878
